# Phase 10 — Prediction API

This engineering notebook exercises the FastAPI service in-process. It displays the health contract, one complete two-model assessment, and a structured validation failure without starting a public server or beginning frontend work.

In [1]:
from pathlib import Path
import json, os, sys
project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))
from fastapi.testclient import TestClient
from IPython.display import JSON, display
from api.app.main import app
from finaccess_eswatini.phase10_api_validation import run
summary = run()
payload = json.loads((project_root / 'api' / 'examples' / 'assessment_request.json').read_text(encoding='utf-8'))
client = TestClient(app)
print('Phase:', summary['phase'], 'Status:', summary['status'])
print('Validated contract scenarios:', summary['validation_cases_passed'], '/', summary['validation_cases'])

C:\Users\Thando F Dlamini\Documents\FinAccess Eswatini\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Phase: 10 Status: PASS_WITH_NOTES
Validated contract scenarios: 8 / 8


## Artifact-aware health check

A healthy response means both pipelines and both SHAP explainers loaded with their validated hashes.

In [2]:
health = client.get('/health')
print('HTTP status:', health.status_code)
display(JSON(health.json()))

HTTP status: 200


<IPython.core.display.JSON object>

## One profile, two explained predictions

The percentage supports a direct natural-language answer. Every listed factor comes from the corresponding persisted SHAP explainer.

In [3]:
response = client.post('/api/v1/assessment', json=payload)
body = response.json()
print('HTTP status:', response.status_code)
for key in ('financial_inclusion', 'mobile_money_adoption'):
    result = body[key]
    print('\n' + result['question'])
    print(result['answer'], f"Estimated likelihood: {result['probability_percent']:.1f}%")
    for factor in result['main_factors']:
        print(' •', factor['explanation'])
display(JSON(body))

HTTP status: 200

Based on the characteristics provided, is this person likely to be financially included?
This person is unlikely to be financially included. Estimated likelihood: 26.9%
 • Education level (Primary education or less) reduced the prediction relative to the model baseline.
 • Workforce status (Out of the workforce) reduced the prediction relative to the model baseline.
 • Age group (65+) increased the prediction relative to the model baseline.
 • Recent internet use (No / don't know / refused) reduced the prediction relative to the model baseline.
 • Income quintile (Income quintile 2) reduced the prediction relative to the model baseline.

Based on the characteristics provided, is this person likely to use mobile money?
This person is unlikely to use mobile money. Estimated likelihood: 36.8%
 • Internet engagement (No recent internet use / no-DK-ref) reduced the prediction relative to the model baseline.
 • Education level (Primary education or less) reduced the predict

<IPython.core.display.JSON object>

## Structured input rejection and documented contract

Unknown categories and contradictory routed fields are rejected before inference.

In [4]:
invalid = dict(payload, internet_engagement_level='Daily internet use')
error = client.post('/api/v1/assessment', json=invalid)
print('Contradictory input status:', error.status_code)
display(JSON(error.json()))
contract = client.get('/openapi.json').json()
print('Documented paths:')
for path, methods in contract['paths'].items():
    print(path, sorted(method.upper() for method in methods))

Contradictory input status: 422


<IPython.core.display.JSON object>

Documented paths:
/ ['GET']
/health ['GET']
/api/v1/assessment ['POST']
